# 08 · GRDECL export

Writes the corner-point grid of notebook 03 with the properties of notebook 04 as an Eclipse-style GRDECL file
(SPECGRID, COORD, ZCORN, ACTNUM and property keywords), then reads it back. This is the grid and properties
only, not a full simulation deck: add RUNSPEC, PROPS, SOLUTION and SCHEDULE sections yourself to run it in a
simulator such as OPM Flow.

## Setup

In [1]:
import os
from pathlib import Path
import numpy as np, pandas as pd, pyarrow.parquet as pq
import matplotlib.pyplot as plt
%matplotlib inline
# SX_EXAMPLE_PROJECT: KGS project folder (las/, edges/, project.toml), see scripts/fetch_open_data.sh.
# SX_EXAMPLE_OUT: folder for figures and the small model files passed between notebooks.
PROJ = Path(os.environ.get("SX_EXAMPLE_PROJECT", "kgs_project")).resolve()
OUT = Path(os.environ.get("SX_EXAMPLE_OUT", "outputs")).resolve()
OUT.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 80, "savefig.dpi": 110})
def save(fig, name):
    fig.savefig(OUT / name, bbox_inches="tight")

## Build the grid and properties

In [2]:
from subsurfix import _native as nat
g, mdl = np.load(OUT / "grid_upscaled.npz"), np.load(OUT / "property_model.npz")
nz = int(g["dims"][2]); dz = float(g["cell"][2])
cpg = nat.CornerPointGrid.from_surfaces(g["xs"], g["ys"], np.array([g["top"] + dz * k for k in range(nz + 1)]))
phi = np.clip(mdl["phi"], 0.01, 0.35)
perm = 10 ** (-1.0 + 14.0 * phi) * np.where(mdl["clean"] == 1, 2.0, 1.0)      # mD, illustrative
props = {"PORO": phi.tolist(), "PERMX": perm.tolist(), "PERMY": perm.tolist(), "PERMZ": (0.1 * perm).tolist()}
print(cpg.dims, "cells; porosity mean", phi.mean().round(3))

(22, 22, 12) cells; porosity mean 0.13


## Write and read back

In [3]:
text = cpg.to_grdecl(props)
(OUT / "model.grdecl").write_text(text)
back, bprops = nat.CornerPointGrid.from_grdecl(text)
print("wrote", round(len(text) / 1e6, 2), "MB; keywords read back:", sorted(bprops))
print("geometry identical:", np.allclose(back.zcorn, cpg.zcorn) and np.allclose(back.coord, cpg.coord),
      "| PORO identical:", np.allclose(bprops["PORO"], phi, atol=1e-6))
print(text[:300])

wrote 1.34 MB; keywords read back: ['PERMX', 'PERMY', 'PERMZ', 'PORO']
geometry identical: True | PORO identical: True
-- Written by SubsurfiX
SPECGRID
 22 22 12 1 F /

COORD
-4500 -5250 954.1496322045729 -4500 -5250 1089.504339860512
-4000 -5250 954.1496322045729 -4000 -5250 1089.504339860512
-3500 -5250 954.1496322045729 -3500 -5250 1089.504339860512
-3000 -5250 954.1496322045729 -3000 -5250 1089.504339860512
-250
